<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/Codex-Branch/TwoAgents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone -b Codex-Branch https://github.com/vishaljoshi24/DungeonsAndDragonsTurnClassification/
%cd /content/DungeonsAndDragonsTurnClassification

fatal: destination path 'DungeonsAndDragonsTurnClassification' already exists and is not an empty directory.
/content/DungeonsAndDragonsTurnClassification


In [2]:
!pip install dspy

In [3]:
import argparse

In [4]:
from modules import  (
    AgentProfile,
    MultiAgentConversation,
    TwoAgentConversation,
    configure_lm,
    PromptTool,
    InstructedChatbot
)

In [5]:
DEFAULT_SCENARIO = (
    'This D&D short scenario specifically concerns a rat infestation. '
'It is set in the craft brewery Company and is in dire need of help. '
'The players in this scenario are Alice and Bob.'
'The non-player character is Charlie, and he is controlled by the Dungeon Master. '
'Alice and Bob are here to sort out a RAT INFESTATION in the brewery\'s BASEMENT. '
'For the duration of the scenario, only Alice, Bob and the brewery owner Charlie are in the brewery '
'At the beginning of this adventure Alice and Bob '
'meet in the Wizard Tower Brewing Company. These two adventurers '
'DO NOT know each other AT FIRST and need to get to know each other. '
'Charlie hands out pints of Ale to Alice and Bob as they get to know each other. '
# 'Alice and Bob should start by asking Charlie for some information about the infestation. '
# 'Alice and Bob should then come up with a strategy for clearing out the infestation. '
# 'Finally, Alice and Bob should explore the brewery\'s cellar based on their strategy. '
)

In [6]:
instructions_path = "instructions.xlsx"
prompt_tool = PromptTool(prompts_path=instructions_path)

In [7]:
def build_conversation() -> MultiAgentConversation:
    alice = AgentProfile(
        name="Alice",
        character="Alice, a dwarf barbarian.",
        instructions=(
        'The instructions for how to play the role of a D&D player are as '
        'follows. This is a short scenario in which you '
        'play the role of a character named Alice. This scenario '
        'is structured as a Dungeons & Dragons game. '
        'The goal is to be consistent, but creative. It is '
        'important to play the role of a Dungeons & Dragons player as '
        'accurately as possible, i.e., by responding in ways that you think '
        'it is likely a player would respond, and taking '
        'into account all information that you have. '
        'It is important that you collaborate with the  '
        'other player, on the task at hand and listen to the Dungeon Master\'s '
        'insturctions.'
        'Do not be verbose. '
        'Do not repeat yourself. '
        'Always use first-person limited perspective.'
        ),
        inventory = [
                "One set of common clothes",
                "Two daggers",
                "One axe",
                "50 feet of rope",
                "One tinderbox",
                "One torch",
            ],
        abilities = {""},
        skills = {""},
    )
    bob = AgentProfile(
        name="Bob",
        character="Bob, a wizard.",
        instructions=(
        'The instructions for how to play the role of a D&D player are as '
        'follows. This is a short scenario in which you '
        'play the role of a character named Alice. This scenario '
        'is structured as a Dungeons & Dragons game. '
        'The goal is to be consistent, but creative. It is '
        'important to play the role of a Dungeons & Dragons player as '
        'accurately as possible, i.e., by responding in ways that you think '
        'it is likely a player would respond, and taking '
        'into account all information that you have. '
        'It is important that you collaborate with the  '
        'other player, on the task at hand and listen to the Dungeon Master\'s '
        'insturctions.'
        'Do not be verbose. '
        'Do not repeat yourself. '
        'Always use first-person limited perspective.'
        ),
        inventory = [
                "One Wizard's Staff",
                "One can of oil",
                "One tinderbox",
                "Thunderwave spell: You unleash a wave of thunderous energy.",
                "Command spell: You speak a one-word command to a creature you can see within range."
            ],
        abilities = {""},
        skills = {""},
    )
    dungeon_master = AgentProfile(
          name="Dungeon Master",
          character="The Dungeon Master, who also plays the role of Charlie: the owner of the Wizards Tower Brewing Company.",
          instructions= (
          'This is a tabletop role-playing game: Dungeons & Dragons. You are the Dungeon Master.'
          'You will describe the current situation to the players in the game and then on the basis '
          'of what you tell them they will suggest actions for the character they control. '
          'You will then decide if the action is valid based on Dungeons & Dragons 5th Edition rules. '
          'Aside from you, each other player controls just one character. '
          'If any of the players deviates dramatically from the scenario your response should attempt to re-orient the scenario. '
          'You are the Dungeon Master so you control any non-player characters or adversaries. '
          'You can answer player\'s questions and give them any items they need, but you should '
          'not get involved with the task directly. '
          'You will track the state of the world and keep it consistent as time  '
          'passes in the simulation and the players take actions and change things in their world. '
          'Remember that this is a game. It should be fun for the players. '
          'You should use second-person perspective, when speaking directly to the players. '
          'You should use first-person limited perspective when role-playing as non-player characters and adversaries.'
          'End the scenario before players start combat.'
          'Do not be verbose. Do not repeat yourself.'
          ),
          inventory = [""],
          abilities = {""},
          skills = {""},
    )

    return MultiAgentConversation(
        agent_a=alice,
        agent_b=bob,
        agent_c=dungeon_master,
        scenario=DEFAULT_SCENARIO,
        chatbot_a=InstructedChatbot(prompt_tool=prompt_tool),
        chatbot_b=InstructedChatbot(prompt_tool=prompt_tool),
        chatbot_c=InstructedChatbot(prompt_tool=prompt_tool)
    )

In [8]:
parser = argparse.ArgumentParser(description="Run three D&D DSPy agents together.")

In [9]:
parser.add_argument(
    "--opening-message",
    default="You find yourselves in the main taproom of the brewery, questioning Charlie the brewery owner.",
    help="The first turn spoken by the Dungeon Master.",
)

_StoreAction(option_strings=['--opening-message'], dest='opening_message', nargs=None, const=None, default='You find yourselves in the main taproom of the brewery, questioning Charlie the brewery owner.', type=None, choices=None, required=False, help='The first turn spoken by the Dungeon Master.', metavar=None)

In [10]:
parser.add_argument(
    "--rounds",
    type=int,
    default=15,
    help="Number of replies to generate after the opening message.",
)

_StoreAction(option_strings=['--rounds'], dest='rounds', nargs=None, const=None, default=15, type=<class 'int'>, choices=None, required=False, help='Number of replies to generate after the opening message.', metavar=None)

In [11]:
parser.add_argument("--model", default="ollama_chat/qwen3:8b")

_StoreAction(option_strings=['--model'], dest='model', nargs=None, const=None, default='ollama_chat/qwen3:8b', type=None, choices=None, required=False, help=None, metavar=None)

In [12]:
parser.add_argument("--api-base", default="http://localhost:11434")

_StoreAction(option_strings=['--api-base'], dest='api_base', nargs=None, const=None, default='http://localhost:11434', type=None, choices=None, required=False, help=None, metavar=None)

In [13]:
args = parser.parse_args([])

In [14]:
configure_lm(model=args.model, api_base=args.api_base)

In [15]:
conversation = build_conversation()

In [16]:
for turn in conversation.run(args.opening_message, rounds=args.rounds):
    print(turn)

Dungeon Master: You find yourselves in the main taproom of the brewery, questioning Charlie the brewery owner.
Alice: “Where’s the rat nest? We’re here to fix this.”
Bob: "The basement, then. I’ll handle the vermin once we’re there—no need for brute force."
Dungeon Master: "Ah, a clever approach! The basement stairs are through the back door—follow me. The rats have made a mess of the grain storage, but with your skills, this should be manageable. Let’s move!"
Alice: "Stay close, Bob. I’ll clear the path—don’t let the rats get the upper hand."
Bob: "Good. Let them come—my thunderwave will scatter them like dust."
Dungeon Master: The thunderwave cracks the air, sending the rats scurrying. They swarm into the grain storage, creating a chaotic mess. Now, you must navigate the piled grain while dealing with the rats’ relentless advance.
Alice: "Keep your distance, Bob. I’ll split the pack—axe through the thick of them. Rope’s for trapping, not wrestling."
Bob: I'll carve a path through the

In [17]:
import dspy
dspy.inspect_history(n=10)





[2026-07-23T14:49:15.545301]

System message:

Your input fields are:
1. `instruct_prompt` (str): 
2. `scenario_context` (str): 
3. `user_input` (str): 
4. `trajectory` (str):
Your output fields are:
1. `next_thought` (str): 
2. `next_tool_name` (Literal['classify_input', 'search_prompts', 'update_prompt', 'finish']): 
3. `next_tool_args` (dict[str, Any]):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## instruct_prompt ## ]]
{instruct_prompt}

[[ ## scenario_context ## ]]
{scenario_context}

[[ ## user_input ## ]]
{user_input}

[[ ## trajectory ## ]]
{trajectory}

[[ ## next_thought ## ]]
{next_thought}

[[ ## next_tool_name ## ]]
{next_tool_name}        # note: the value you produce must exactly match (no extra characters) one of: classify_input; search_prompts; update_prompt; finish

[[ ## next_tool_args ## ]]
{next_tool_args}        # note: the value you produce must adhere to the JSON schema: {"type": "object", "additional